<a href="https://colab.research.google.com/github/rutujagawade-data/wholesale-banking-sales-pipeline-dashboard/blob/main/generate_sales_pipeline_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

folders = [
    "data/raw",
    "data/processed"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [2]:
# -----------------------------
# Reference Data
# -----------------------------

industries = [
    "Technology",
    "Healthcare",
    "Manufacturing",
    "Real Estate",
    "Energy",
    "Retail",
    "Financial Services",
    "Transportation"
]

regions = [
    "Southeast",
    "Northeast",
    "Midwest",
    "West"
]

segments = [
    "Middle Market",
    "Large Corporate",
    "Commercial",
    "Strategic"
]

# -----------------------------
# Clients
# -----------------------------

clients = pd.DataFrame({
    "client_id": [f"CL-{str(i).zfill(4)}" for i in range(1, 151)],
    "client_name": [f"Client {i}" for i in range(1, 151)],
    "industry": np.random.choice(industries, 150),
    "region": np.random.choice(regions, 150),
    "client_segment": np.random.choice(segments, 150),
    "relationship_start_date": pd.to_datetime(
        np.random.choice(pd.date_range("2018-01-01", "2024-12-31"), 150)
    )
})

# -----------------------------
# Products
# -----------------------------

products = pd.DataFrame({
    "product_id": ["PR-001", "PR-002", "PR-003", "PR-004", "PR-005", "PR-006"],
    "product_name": [
        "Commercial Loan",
        "Treasury Management",
        "Capital Markets",
        "M&A Advisory",
        "Credit Facility",
        "Equipment Finance"
    ],
    "product_group": [
        "Lending",
        "Treasury",
        "Capital Markets",
        "Advisory",
        "Lending",
        "Lending"
    ],
    "fee_type": [
        "Interest / Fee",
        "Service Fee",
        "Transaction Fee",
        "Advisory Fee",
        "Commitment Fee",
        "Financing Fee"
    ]
})

# -----------------------------
# Relationship Managers
# -----------------------------

relationship_managers = pd.DataFrame({
    "rm_id": [f"RM-{str(i).zfill(3)}" for i in range(1, 26)],
    "relationship_manager": [
        "Avery Johnson", "Morgan Smith", "Taylor Brown", "Jordan Davis", "Casey Miller",
        "Riley Wilson", "Jamie Moore", "Cameron Taylor", "Drew Anderson", "Quinn Thomas",
        "Skyler Jackson", "Parker White", "Reese Harris", "Alex Martin", "Blake Thompson",
        "Hayden Garcia", "Logan Martinez", "Emerson Robinson", "Finley Clark", "Rowan Lewis",
        "Harper Walker", "Sage Young", "Dakota Allen", "Kendall King", "Marley Scott"
    ],
    "team": np.random.choice(["Team A", "Team B", "Team C", "Team D"], 25),
    "region": np.random.choice(regions, 25),
    "manager_name": np.random.choice(["Director 1", "Director 2", "Director 3"], 25)
})

# -----------------------------
# Targets
# -----------------------------

quarters = ["2026-Q1", "2026-Q2"]
product_groups = products["product_group"].unique()

target_rows = []
target_id = 1

for quarter in quarters:
    for region in regions:
        for product_group in product_groups:
            target_rows.append({
                "target_id": f"TGT-{str(target_id).zfill(4)}",
                "quarter": quarter,
                "region": region,
                "product_group": product_group,
                "target_fee": np.random.randint(750000, 2000000),
                "target_pipeline": np.random.randint(7500000, 20000000)
            })
            target_id += 1

targets = pd.DataFrame(target_rows)

# Save raw files
clients.to_csv("data/raw/clients.csv", index=False)
products.to_csv("data/raw/products.csv", index=False)
relationship_managers.to_csv("data/raw/relationship_managers.csv", index=False)
targets.to_csv("data/raw/targets.csv", index=False)

print("Base raw CSV files created.")
print("clients:", clients.shape)
print("products:", products.shape)
print("relationship_managers:", relationship_managers.shape)
print("targets:", targets.shape)

Base raw CSV files created.
clients: (150, 6)
products: (6, 4)
relationship_managers: (25, 5)
targets: (32, 6)


In [3]:
# -----------------------------
# Opportunities
# -----------------------------

stages = [
    "Prospecting",
    "Qualification",
    "Proposal",
    "Negotiation",
    "Commitment",
    "Closed Won",
    "Closed Lost"
]

stage_probabilities = {
    "Prospecting": 0.10,
    "Qualification": 0.25,
    "Proposal": 0.50,
    "Negotiation": 0.70,
    "Commitment": 0.90,
    "Closed Won": 1.00,
    "Closed Lost": 0.00
}

open_stages = [
    "Prospecting",
    "Qualification",
    "Proposal",
    "Negotiation",
    "Commitment"
]

num_opportunities = 750

opportunities = pd.DataFrame({
    "opportunity_id": [f"OPP-{str(i).zfill(5)}" for i in range(1, num_opportunities + 1)],
    "client_id": np.random.choice(clients["client_id"], num_opportunities),
    "product_id": np.random.choice(products["product_id"], num_opportunities),
    "rm_id": np.random.choice(relationship_managers["rm_id"], num_opportunities),
    "created_date": pd.to_datetime(
        np.random.choice(pd.date_range("2025-09-01", "2026-03-15"), num_opportunities)
    ),
    "expected_close_date": pd.to_datetime(
        np.random.choice(pd.date_range("2026-01-15", "2026-06-30"), num_opportunities)
    ),
    "opportunity_amount": np.random.randint(250000, 10000000, num_opportunities),
    "source_channel": np.random.choice(
        ["Relationship Coverage", "Referral", "Event", "Inbound", "Product Partner"],
        num_opportunities
    )
})

# Expected fee as a percentage of opportunity amount
opportunities["expected_fee"] = (
    opportunities["opportunity_amount"] * np.random.uniform(0.005, 0.025, num_opportunities)
).round(2)

# Latest stage/status
opportunities["stage"] = np.random.choice(
    stages,
    num_opportunities,
    p=[0.18, 0.20, 0.22, 0.18, 0.10, 0.07, 0.05]
)

opportunities["status"] = np.where(
    opportunities["stage"].isin(["Closed Won", "Closed Lost"]),
    opportunities["stage"],
    "Open"
)

# Actual close date only for closed opportunities
random_close_dates = pd.to_datetime(
    np.random.choice(pd.date_range("2026-01-01", "2026-03-31"), num_opportunities)
)

opportunities["actual_close_date"] = np.where(
    opportunities["status"].isin(["Closed Won", "Closed Lost"]),
    random_close_dates,
    pd.NaT
)

opportunities.to_csv("data/raw/opportunities.csv", index=False)

print("opportunities.csv created.")
print("opportunities:", opportunities.shape)
opportunities.head()

opportunities.csv created.
opportunities: (750, 12)


,opportunity_id,client_id,product_id,rm_id,created_date,expected_close_date,opportunity_amount,source_channel,expected_fee,stage,status,actual_close_date
0,OPP-00001,CL-0133,PR-002,RM-022,2025-11-21,2026-05-29,987090,Inbound,7544.91,Prospecting,Open,NaT
1,OPP-00002,CL-0075,PR-006,RM-021,2025-11-08,2026-05-03,9553173,Referral,131806.21,Commitment,Open,NaT
2,OPP-00003,CL-0146,PR-001,RM-022,2026-01-09,2026-06-26,4085598,Inbound,73046.94,Qualification,Open,NaT
3,OPP-00004,CL-0076,PR-004,RM-009,2025-11-26,2026-03-03,5232042,Inbound,66023.33,Qualification,Open,NaT
4,OPP-00005,CL-0009,PR-005,RM-009,2026-02-12,2026-03-28,1302425,Event,7933.47,Proposal,Open,NaT


In [4]:
# -----------------------------
# Weekly Pipeline Snapshots
# -----------------------------

snapshot_dates = pd.date_range("2026-01-03", "2026-03-28", freq="W-SAT")

snapshot_rows = []

for _, opp in opportunities.iterrows():
    created_date = pd.to_datetime(opp["created_date"])
    expected_close_date = pd.to_datetime(opp["expected_close_date"])

    # Opportunity appears only in snapshots after created date
    active_snapshots = [d for d in snapshot_dates if d >= created_date]

    if len(active_snapshots) == 0:
        continue

    # Start most opportunities in early stages
    current_stage_idx = np.random.randint(0, 3)

    for snapshot_date in active_snapshots:

        # Some opportunities progress stage over time
        if np.random.rand() < 0.20 and current_stage_idx < len(open_stages) - 1:
            current_stage_idx += 1

        stage = open_stages[current_stage_idx]

        # Some opportunities close near/after expected close date
        if snapshot_date >= expected_close_date and np.random.rand() < 0.22:
            stage = np.random.choice(["Closed Won", "Closed Lost"], p=[0.65, 0.35])

        probability = stage_probabilities[stage]
        status = stage if stage in ["Closed Won", "Closed Lost"] else "Open"

        days_since_last_activity = np.random.randint(1, 45)
        is_stale = status == "Open" and days_since_last_activity > 21
        is_past_due = status == "Open" and expected_close_date < snapshot_date

        expected_fee = float(opp["expected_fee"])
        weighted_fee = expected_fee * probability if status == "Open" else 0

        snapshot_rows.append({
            "snapshot_id": f"SNP-{opp['opportunity_id']}-{snapshot_date.strftime('%Y%m%d')}",
            "snapshot_date": snapshot_date,
            "opportunity_id": opp["opportunity_id"],
            "stage": stage,
            "status": status,
            "opportunity_amount": opp["opportunity_amount"],
            "expected_fee": expected_fee,
            "probability": probability,
            "weighted_fee": round(weighted_fee, 2),
            "days_since_last_activity": days_since_last_activity,
            "is_stale": is_stale,
            "is_past_due": is_past_due,
            "expected_close_date": expected_close_date
        })

        # Stop future snapshots after opportunity closes
        if stage in ["Closed Won", "Closed Lost"]:
            break

weekly_pipeline_snapshots = pd.DataFrame(snapshot_rows)

weekly_pipeline_snapshots.to_csv("data/raw/weekly_pipeline_snapshots.csv", index=False)

print("weekly_pipeline_snapshots.csv created.")
print("weekly_pipeline_snapshots:", weekly_pipeline_snapshots.shape)
weekly_pipeline_snapshots.head()

weekly_pipeline_snapshots.csv created.
weekly_pipeline_snapshots: (7353, 13)


,snapshot_id,snapshot_date,opportunity_id,stage,status,opportunity_amount,expected_fee,probability,weighted_fee,days_since_last_activity,is_stale,is_past_due,expected_close_date
0,SNP-OPP-00001-20260103,2026-01-03,OPP-00001,Prospecting,Open,987090,7544.91,0.1,754.49,25,True,False,2026-05-29
1,SNP-OPP-00001-20260110,2026-01-10,OPP-00001,Prospecting,Open,987090,7544.91,0.1,754.49,39,True,False,2026-05-29
2,SNP-OPP-00001-20260117,2026-01-17,OPP-00001,Prospecting,Open,987090,7544.91,0.1,754.49,32,True,False,2026-05-29
3,SNP-OPP-00001-20260124,2026-01-24,OPP-00001,Prospecting,Open,987090,7544.91,0.1,754.49,39,True,False,2026-05-29
4,SNP-OPP-00001-20260131,2026-01-31,OPP-00001,Prospecting,Open,987090,7544.91,0.1,754.49,22,True,False,2026-05-29


In [5]:
# -----------------------------
# Build Tableau Mart
# -----------------------------

mart = weekly_pipeline_snapshots.merge(
    opportunities[[
        "opportunity_id",
        "client_id",
        "product_id",
        "rm_id",
        "created_date",
        "source_channel"
    ]],
    on="opportunity_id",
    how="left"
)

mart = mart.merge(
    clients,
    on="client_id",
    how="left"
)

mart = mart.merge(
    products,
    on="product_id",
    how="left"
)

mart = mart.merge(
    relationship_managers[[
        "rm_id",
        "relationship_manager",
        "team",
        "manager_name"
    ]],
    on="rm_id",
    how="left"
)

mart["snapshot_date"] = pd.to_datetime(mart["snapshot_date"])
mart["snapshot_week"] = mart["snapshot_date"]
mart["year"] = mart["snapshot_date"].dt.year
mart["quarter"] = mart["snapshot_date"].dt.to_period("Q").astype(str).str.replace("Q", "-Q")
mart["month"] = mart["snapshot_date"].dt.month
mart["week_number"] = mart["snapshot_date"].dt.isocalendar().week.astype(int)

# KPI helper fields
mart["open_pipeline"] = np.where(
    mart["status"] == "Open",
    mart["opportunity_amount"],
    0
)

mart["open_expected_fee"] = np.where(
    mart["status"] == "Open",
    mart["expected_fee"],
    0
)

mart["stale_expected_fee"] = np.where(
    mart["is_stale"] == True,
    mart["expected_fee"],
    0
)

mart["closed_won_fee"] = np.where(
    mart["status"] == "Closed Won",
    mart["expected_fee"],
    0
)

mart["deal_age_days"] = (
    mart["snapshot_date"] - pd.to_datetime(mart["created_date"])
).dt.days

mart["late_stage_flag"] = mart["stage"].isin(["Negotiation", "Commitment"])

mart["priority_follow_up_flag"] = np.where(
    (mart["is_stale"] == True) | (mart["is_past_due"] == True),
    True,
    False
)

# Add targets by quarter, region, and product group
mart = mart.merge(
    targets[[
        "quarter",
        "region",
        "product_group",
        "target_fee",
        "target_pipeline"
    ]],
    on=["quarter", "region", "product_group"],
    how="left"
)

# Sort for Tableau-friendly trend analysis
mart = mart.sort_values(
    by=["snapshot_date", "region", "product_group", "relationship_manager", "opportunity_id"]
)

mart.to_csv("data/processed/mart_pipeline_weekly.csv", index=False)

print("mart_pipeline_weekly.csv created.")
print("mart_pipeline_weekly:", mart.shape)
mart.head()

mart_pipeline_weekly.csv created.
mart_pipeline_weekly: (7353, 43)


,snapshot_id,snapshot_date,opportunity_id,stage,status,opportunity_amount,expected_fee,probability,weighted_fee,days_since_last_activity,...,week_number,open_pipeline,open_expected_fee,stale_expected_fee,closed_won_fee,deal_age_days,late_stage_flag,priority_follow_up_flag,target_fee,target_pipeline
3825,SNP-OPP-00390-20260103,2026-01-03,OPP-00390,Qualification,Open,4696460,55889.46,0.25,13972.36,32,...,1,4696460,55889.46,55889.46,0.0,53,False,True,1448646,10234296
5226,SNP-OPP-00533-20260103,2026-01-03,OPP-00533,Proposal,Open,6341111,74164.24,0.50,37082.12,36,...,1,6341111,74164.24,74164.24,0.0,80,False,True,1448646,10234296
2988,SNP-OPP-00304-20260103,2026-01-03,OPP-00304,Qualification,Open,7517807,58232.94,0.25,14558.24,44,...,1,7517807,58232.94,58232.94,0.0,95,False,True,1448646,10234296
3767,SNP-OPP-00383-20260103,2026-01-03,OPP-00383,Qualification,Open,2684138,53796.26,0.25,13449.07,36,...,1,2684138,53796.26,53796.26,0.0,39,False,True,1448646,10234296
3894,SNP-OPP-00397-20260103,2026-01-03,OPP-00397,Proposal,Open,8753197,191498.84,0.50,95749.42,17,...,1,8753197,191498.84,0.00,0.0,113,False,False,1448646,10234296


In [6]:
# -----------------------------
# Validation Summary
# -----------------------------

summary = {
    "clients.csv": clients.shape[0],
    "products.csv": products.shape[0],
    "relationship_managers.csv": relationship_managers.shape[0],
    "targets.csv": targets.shape[0],
    "opportunities.csv": opportunities.shape[0],
    "weekly_pipeline_snapshots.csv": weekly_pipeline_snapshots.shape[0],
    "mart_pipeline_weekly.csv": mart.shape[0]
}

summary_df = pd.DataFrame(list(summary.items()), columns=["file_name", "row_count"])

summary_df

,file_name,row_count
0,clients.csv,150
1,products.csv,6
2,relationship_managers.csv,25
3,targets.csv,32
4,opportunities.csv,750
5,weekly_pipeline_snapshots.csv,7353
6,mart_pipeline_weekly.csv,7353


In [7]:
# -----------------------------
# KPI Validation
# -----------------------------

latest_snapshot = mart["snapshot_date"].max()
latest = mart[mart["snapshot_date"] == latest_snapshot]

open_pipeline = latest["open_pipeline"].sum()
weighted_fee = latest["weighted_fee"].sum()
closed_won_fee = latest["closed_won_fee"].sum()
stale_pipeline_pct = (
    latest["stale_expected_fee"].sum() / latest["open_expected_fee"].sum()
    if latest["open_expected_fee"].sum() != 0
    else 0
)

print("Latest Snapshot:", latest_snapshot.date())
print("Open Pipeline:", round(open_pipeline, 2))
print("Weighted Fee:", round(weighted_fee, 2))
print("Closed Won Fee:", round(closed_won_fee, 2))
print("Stale Pipeline %:", round(stale_pipeline_pct * 100, 2))

Latest Snapshot: 2026-03-28
Open Pipeline: 2787711367
Weighted Fee: 27462553.54
Closed Won Fee: 999574.92
Stale Pipeline %: 51.5


In [8]:
from google.colab import files
import shutil

shutil.make_archive("sales_pipeline_data", "zip", "data")
files.download("sales_pipeline_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>